# 01 Classical Black-Scholes Validation

Validate prices, put-call parity, and Greeks before adding portfolio or quantum machinery.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
S_grid = np.linspace(12000, 36000, 250)
K = 24000.0
T = 30 / 365
r = 0.065
sigma = 0.18
call_curve = black_scholes_price(S_grid, K, T, r, sigma, "call")
put_curve = black_scholes_price(S_grid, K, T, r, sigma, "put")
plt.figure()
plt.plot(S_grid, call_curve, label="Call")
plt.plot(S_grid, put_curve, label="Put")
plt.axvline(K, color="black", linestyle="--", linewidth=1, label="Strike")
plt.title("Black-Scholes call and put price curves")
plt.xlabel("Index level")
plt.ylabel("Option value")
plt.legend()
save_current_figure("01_black_scholes_call_put_curves.png")
curves = pd.DataFrame({"S": S_grid, "call": call_curve, "put": put_curve})
save_table(curves, "01_black_scholes_price_curves.csv")
curves.head()


In [ ]:
assert abs(put_call_parity_error(24000, K, T, r, sigma)) < 1e-8
for opt in ["call", "put"]:
    g = black_scholes_greeks(24000, K, T, r, sigma, opt)
    fd = finite_difference_greeks(24000, K, T, r, sigma, opt)
    assert abs(g["Delta"] - fd["Delta"]) < 1e-4
    assert abs(g["Gamma"] - fd["Gamma"]) < 1e-5
print("VALIDATION PASSED: put-call parity and Greeks comparison")


In [ ]:
greek_rows = []
for S in S_grid:
    row = {"S": S}
    for opt in ["call", "put"]:
        greeks = black_scholes_greeks(float(S), K, T, r, sigma, opt)
        for key, val in greeks.items():
            row[f"{opt}_{key}"] = val
    greek_rows.append(row)
greeks_df = pd.DataFrame(greek_rows)
save_table(greeks_df, "01_analytical_greeks_curves.csv")
for greek in ["Delta", "Gamma", "Vega", "Theta", "Rho"]:
    plt.figure()
    plt.plot(greeks_df["S"], greeks_df[f"call_{greek}"], label=f"Call {greek}")
    plt.plot(greeks_df["S"], greeks_df[f"put_{greek}"], label=f"Put {greek}")
    plt.axvline(K, color="black", linestyle="--", linewidth=1)
    plt.title(f"Analytical {greek} curves")
    plt.xlabel("Index level")
    plt.ylabel(greek)
    plt.legend()
    save_current_figure(f"01_analytical_{greek.lower()}_curves.png")
greeks_df.head()
